# Urease-Quercetin Docking v2
AutoDock Vina | Shayan Asadi

In [ ]:
# STEP 1: Install
!pip install -q rdkit biopython
!apt-get install -q -y autodock-vina openbabel
print("Done")

In [ ]:
# STEP 2: Download receptor
import urllib.request
urllib.request.urlretrieve("https://files.rcsb.org/download/4H9M.pdb", "4H9M.pdb")
print("Downloaded 4H9M.pdb")

In [ ]:
# STEP 3: Clean receptor - keep only protein chain A
lines_out = []
with open("4H9M.pdb") as f:
    for line in f:
        if line.startswith("ATOM") and line[21] == "A":
            lines_out.append(line)
        elif line.startswith("END"):
            lines_out.append(line)
with open("receptor_clean.pdb", "w") as f:
    f.writelines(lines_out)
print(f"Kept {len(lines_out)} ATOM lines")

In [ ]:
# STEP 4: Convert receptor to PDBQT
import subprocess
r = subprocess.run(
    ["obabel", "receptor_clean.pdb", "-O", "receptor.pdbqt",
     "--partialcharge", "gasteiger", "-xr"],
    capture_output=True, text=True)
print(r.stdout, r.stderr)
print("receptor.pdbqt ready")

In [ ]:
# STEP 5: Prepare quercetin ligand
from rdkit import Chem
from rdkit.Chem import AllChem
import subprocess

smiles = "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12"
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
AllChem.MMFFOptimizeMolecule(mol)

w = Chem.SDWriter("quercetin.sdf")
w.write(mol)
w.close()

r = subprocess.run(
    ["obabel", "quercetin.sdf", "-O", "quercetin.pdbqt",
     "--partialcharge", "gasteiger", "-h"],
    capture_output=True, text=True)
print(r.stdout, r.stderr)
print("quercetin.pdbqt ready")

In [ ]:
# STEP 6: Find Ni2+ active site center
with open("4H9M.pdb") as f:
    lines = f.readlines()

ni = []
for line in lines:
    if line.startswith("HETATM") and "NI" in line[12:16]:
        x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
        ni.append((x,y,z))
        print(f"Ni2+: {x:.2f}, {y:.2f}, {z:.2f}")

cx = sum(c[0] for c in ni)/len(ni)
cy = sum(c[1] for c in ni)/len(ni)
cz = sum(c[2] for c in ni)/len(ni)
print(f"Center: {cx:.2f}, {cy:.2f}, {cz:.2f}")

In [ ]:
# STEP 7: Run Vina docking
import subprocess

# Use Ni2+ center from step 6
cx, cy, cz = 18.78, -57.81, -24.15

r = subprocess.run([
    "vina",
    "--receptor", "receptor.pdbqt",
    "--ligand", "quercetin.pdbqt",
    "--center_x", str(cx),
    "--center_y", str(cy),
    "--center_z", str(cz),
    "--size_x", "20",
    "--size_y", "20",
    "--size_z", "20",
    "--exhaustiveness", "16",
    "--num_modes", "9",
    "--out", "docked.pdbqt"
], capture_output=True, text=True)

print(r.stdout)
print(r.stderr)

with open("docking_log.txt","w") as f:
    f.write(r.stdout + r.stderr)

In [ ]:
# STEP 8: Plot results
import re, pandas as pd
import matplotlib.pyplot as plt

with open("docking_log.txt") as f:
    log = f.read()
print(log)

pattern = r"(\d+)\s+(-?\d+\.\d+)\s+(\d+\.\d+)\s+(\d+\.\d+)"
matches = re.findall(pattern, log)

if matches:
    df = pd.DataFrame(matches, columns=["Mode","Affinity","RMSD_lb","RMSD_ub"])
    df["Affinity"] = df["Affinity"].astype(float)
    df["Mode"] = df["Mode"].astype(int)
    print(df)

    fig, ax = plt.subplots(figsize=(8,5))
    colors = ["#1D9E75" if i==0 else "#94C4B0" for i in range(len(df))]
    ax.bar(df["Mode"], df["Affinity"], color=colors, edgecolor="white")
    ax.set_xlabel("Mode")
    ax.set_ylabel("Affinity (kcal/mol)")
    ax.set_title(f"Quercetin-Urease Docking | Best: {df.iloc[0]['Affinity']} kcal/mol")
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig("docking_affinities.png", dpi=150)
    plt.show()
    print("Saved: docking_affinities.png")
else:
    print("No results parsed")

In [ ]:
# STEP 9: Download outputs
from google.colab import files
import os
for f in ["docking_affinities.png", "docking_log.txt", "docked.pdbqt"]:
    if os.path.exists(f):
        files.download(f)
        print(f"Downloaded: {f}")